<a href="https://colab.research.google.com/github/jackcase04/scripts/blob/main/NLP_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.9 MB/s eta 0:00:00


In [ ]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-04-18 18:52:59--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-18 18:53:00--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.03MB/s    in 2m 39s  

2026-04-18 18:55:39 (5.17 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflatin

In [ ]:
groups_to_keep = [
    "capital-common-countries",
    "capital-world",
    "currency",
    "city-in-state",
    "family",
    "gram1-adjective-to-adverb",
    "gram2-opposite",
    "gram3-comparative",
    "gram6-nationality-adjective",
    "gram8-plural"
]

with open("mikolov_analogy_dataset.txt") as file:
  current_group = ""
  all_groups = {}

  for line in file:
    start = line[0]

    if start == '/':
      pass
    elif start ==':':
      group = line.split(' ')[1].split('\n')[0]

      if group in groups_to_keep:
        # print(f"Switching to new group: {group}")
        current_group = group

    else:
      if current_group in groups_to_keep:
        analogy = line.strip().split()
        for i in range(len(analogy)):
          analogy[i] = analogy[i].lower()
        # print(f"Analogy: {analogy}")
        if all_groups.get(current_group, 0) == 0:
          all_groups[current_group] = []
        all_groups[current_group].append(analogy)

for group, questions in all_groups.items():
  print(f"{group}: {len(questions)} questions")

capital-common-countries: 506 questions
capital-world: 4524 questions
currency: 866 questions
city-in-state: 2467 questions
family: 506 questions
gram1-adjective-to-adverb: 992 questions
gram2-opposite: 812 questions
gram3-comparative: 3510 questions
gram6-nationality-adjective: 3159 questions
gram8-plural: 2202 questions


In [ ]:
import gensim.downloader as api
model = api.load("glove-wiki-gigaword-100")
print("Model loaded!")

[==================================================] 100.0% 128.1/128.1MB downloaded
Model loaded!


In [ ]:
results = {}
total_correct = 0
total_questions = 0

for group, questions in all_groups.items():
    correct = 0

    for a, b, c, d in questions:
        try:
          predictions = model.most_similar(positive=[b, c], negative=[a], topn=10)
        except:
          continue

        predicted_word = ""
        for word, score in predictions:
            if word != a and word != b and word != c:
                predicted_word = word
                break

        if predicted_word == d:
            correct += 1

    total = len(questions)
    results[group] = {
        'correct': correct,
        'total': total
    }
    total_correct += correct
    total_questions += total

print(results)
print("Done!")

{'capital-common-countries': {'correct': 475, 'total': 506}, 'capital-world': {'correct': 4024, 'total': 4524}, 'currency': {'correct': 123, 'total': 866}, 'city-in-state': {'correct': 760, 'total': 2467}, 'family': {'correct': 413, 'total': 506}, 'gram1-adjective-to-adverb': {'correct': 242, 'total': 992}, 'gram2-opposite': {'correct': 163, 'total': 812}, 'gram3-comparative': {'correct': 2397, 'total': 3510}, 'gram6-nationality-adjective': {'correct': 2270, 'total': 3159}, 'gram8-plural': {'correct': 1467, 'total': 2202}}
Done!


In [ ]:
overall_accuracy = total_correct / total_questions
print(f"Overall accuracy: {overall_accuracy:.3f}")
print("NOTE: Accuracy = precision = recall, for this example\n")

print("Groupwise accuracy:")

for group, value in results.items():
  accuracy = value['correct'] / value['total']
  print(f"Group: {group} accuracy = {accuracy}")


Overall accuracy: 0.631
NOTE: Accuracy = precision = recall, for this example

Groupwise accuracy:
Group: capital-common-countries accuracy = 0.9387351778656127
Group: capital-world accuracy = 0.8894783377541998
Group: currency accuracy = 0.1420323325635104
Group: city-in-state accuracy = 0.3080664775030401
Group: family accuracy = 0.8162055335968379
Group: gram1-adjective-to-adverb accuracy = 0.2439516129032258
Group: gram2-opposite accuracy = 0.20073891625615764
Group: gram3-comparative accuracy = 0.6829059829059829
Group: gram6-nationality-adjective accuracy = 0.7185818296929408
Group: gram8-plural accuracy = 0.6662125340599455
